In [1]:
import numpy as np
import pandas as pd
# from alinea.caribu.CaribuScene import CaribuScene

from openalea.widgets.plantgl import * 
from openalea.archicrop.archicrop import ArchiCrop
from openalea.archicrop.cereal_plant import cereal
# from openalea.archicrop.display import build_scene, display_scene
# from openalea.archicrop.stics_io import read_sti_file, read_xml_file
from openalea.plantgl.all import Color3, Material, Scene, Viewer
from openalea.archicrop.display import build_scene
# from openalea.archicrop.stand import compute_domain
# from openalea.archicrop.stics_io import stics_weather_3d
# from openalea.archicrop.sky_sources import meteo_day
from openalea.astk.sky_irradiance import sky_irradiance
from openalea.astk.sky_sources import caribu_light_sources, sky_sources
from openalea.archicrop.stand import compute_domain
from openalea.archicrop.light_it import illuminate, mean_leaf_irradiance

%gui qt

In [2]:
weather_file = '../data/ntarla_corr.2018'
location = {  
'longitude': 3.87,
'latitude': 0,
'altitude': 800,
'timezone': 'Europe/Paris'}

def meteo_day(filename):
    names=['station', 'year', 'month', 'day', 'julian', 'min_temp', 'max_temp', 'rad', 'Penman PET', 'rainfall', 'wind', 'pressure', 'CO2']
    df = pd.read_csv(filename,  header=None, sep='\s+', names=names)  # noqa: PD901
    df["daydate"] = pd.to_datetime(df[["year", "month", "day"]])
    return df

df = meteo_day(weather_file) 

<>:10: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
C:\Users\cheriere\AppData\Local\Temp\ipykernel_35104\2365886376.py:10: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(filename,  header=None, sep='\s+', names=names)  # noqa: PD901


In [3]:
df

,station,year,month,day,julian,min_temp,max_temp,rad,Penman PET,rainfall,wind,pressure,CO2,daydate
0,ntarla,2018,1,1,1,5.5,32.7,16.2,-999.9,0.0,3.0,9.1,408.72,2018-01-01
1,ntarla,2018,1,2,2,8.4,33.5,15.5,-999.9,0.0,2.8,10.2,408.72,2018-01-02
2,ntarla,2018,1,3,3,6.5,33.5,14.4,-999.9,0.0,2.5,10.7,408.72,2018-01-03
3,ntarla,2018,1,4,4,7.4,33.5,15.4,-999.9,0.0,2.4,12.3,408.72,2018-01-04
4,ntarla,2018,1,5,5,8.0,34.6,17.4,-999.9,0.0,2.4,10.6,408.72,2018-01-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,ntarla,2018,12,27,361,12.0,34.0,16.4,-999.9,0.0,2.0,16.6,408.72,2018-12-27
361,ntarla,2018,12,28,362,13.5,35.5,16.4,-999.9,0.0,1.5,17.2,408.72,2018-12-28
362,ntarla,2018,12,29,363,15.3,36.0,15.2,-999.9,0.0,1.1,20.1,408.72,2018-12-29
363,ntarla,2018,12,30,364,16.5,36.0,14.9,-999.9,0.0,2.0,19.8,408.72,2018-12-30


In [4]:
par = df.rad[0]*0.48
irr = sky_irradiance(daydate='2018-01-01', day_ghi=par, **location)
irr

,azimuth,zenith,elevation,ghi,dni,dhi,ppfd
2018-01-01 07:00:00+01:00,113.027210,87.017596,2.982404,2.096379,0.000000,2.096379,6.459607
2018-01-01 08:00:00+01:00,114.047704,73.422450,16.577550,65.968587,0.000000,65.968587,161.336626
2018-01-01 09:00:00+01:00,116.835752,59.886511,30.113489,144.588749,0.000000,144.588749,335.041278
2018-01-01 10:00:00+01:00,122.371894,46.824076,43.175924,213.306395,5.685984,209.415813,482.286609
2018-01-01 11:00:00+01:00,133.043582,34.887578,55.112422,265.068786,1.555802,263.792598,591.891934
2018-01-01 12:00:00+01:00,153.853251,25.775301,64.224699,295.723314,1.853987,294.053786,656.510370
2018-01-01 13:00:00+01:00,186.987068,23.154321,66.845679,302.971270,4.753796,298.600397,671.767932
2018-01-01 14:00:00+01:00,216.050090,28.859251,61.140749,286.278696,1.768658,284.729691,636.617986
2018-01-01 15:00:00+01:00,232.084288,39.415736,50.584264,246.876700,1.345902,245.836911,553.455856
2018-01-01 16:00:00+01:00,240.290669,51.920409,38.079591,187.771070,2.717174,186.095238,427.869546


On teste si la somme des irradiances de chaque heure est égale à l'irradiance journalière.

In [5]:
round(sum(irr.ghi * 3600)*1e-6, 3) == par

np.True_

In [6]:
sun, sky = sky_sources(sky_type='clear_sky', sky_irradiance=irr, scale='global', force_hi=True)

In [7]:
sun

[]

In [8]:
sky

[(90.0, np.float64(270.0), np.float64(0.24629026134905627)),
 (26.57, np.float64(90.0), np.float64(0.07993600158751898)),
 (26.57, np.float64(18.0), np.float64(0.11400786257584014)),
 (26.57, np.float64(306.0), np.float64(0.18124229807024847)),
 (26.57, np.float64(234.0), np.float64(0.1816182708087322)),
 (26.57, np.float64(162.0), np.float64(0.11423807021537043)),
 (52.62, np.float64(54.0), np.float64(0.14722432337522423)),
 (52.62, np.float64(342.0), np.float64(0.3246520435225561)),
 (52.62, np.float64(270.0), np.float64(0.37427943360869087)),
 (52.62, np.float64(198.0), np.float64(0.32530675655696767)),
 (52.62, np.float64(126.0), np.float64(0.14729868862044387)),
 (10.81, np.float64(54.0), np.float64(0.06469859718127194)),
 (10.81, np.float64(342.0), np.float64(0.1128291331900369)),
 (10.81, np.float64(270.0), np.float64(0.09170553689790588)),
 (10.81, np.float64(198.0), np.float64(0.11370664659771527)),
 (10.81, np.float64(126.0), np.float64(0.06469333519515312)),
 (69.16, np.floa

In [9]:
len(sky)

46

In [10]:
sum([s[2] for s in sky])

np.float64(7.776000000000002)

In [60]:
g = cereal(
        nb_phy=28, phyllochron=30, plastochron=30, stem_duration=2, leaf_duration=2,
        leaf_lifespan=100, leaf_lifespan_early=80, end_juv=50, nb_tillers=0, tiller_delay=2, reduction_factor=1,
        height=150, leaf_area=3500, nb_short_phy=4, short_phy_height=3, wl=0.13,
        diam_base=5.5, diam_top=1.5, insertion_angle=30, scurv=0.50, curvature=60,
        klig=0.6, swmax=0.55, f1=0.64, f2=0.92, stem_q=1.0, rmax=0.7, skew=0.005,
        phyllotactic_angle=180, phyllotactic_deviation=30, tiller_angle=30,
        gravitropism_coefficient=0, plant_orientation=45, spiral=False, classic=False
    )


In [61]:
m = Material(Color3(0,50,0))
scene, _ = build_scene(g, leaf_material = m, stem_material = m)
PlantGL(scene)

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [31]:
density = 5.4
inter_row = 0.4

from openalea.archicrop.stand import agronomic_plot

nplants, positions, domain, domain_area, unit = agronomic_plot(length=2, width=2, density=density, inter_row=inter_row, noise=0)

scene_crop, labels = build_scene(g, positions, leaf_material = m, stem_material = m)
PlantGL(scene_crop)

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [91]:
def getSoilEnergy(cs):
    """ Compute energy received on soil.
    """
    Qi, Einc = None, None

    res = cs.soil_aggregated

    Qi = list(res.values())[0]['Ei']
    Qabs = list(res.values())[0]['Eabs']
    if cs.pattern is not None:
        xmin, ymin, xmax, ymax = cs.pattern
        d_area = abs((xmax - xmin) * (ymax - ymin)) * cs.conv_unit**2
        Einc = Qi * d_area
        Eabs = Qabs * d_area

    return Qi, Einc, Qabs, Eabs

In [92]:
domain = compute_domain(density = density, inter_row = inter_row)
# domain = ((-100, -100), (100, 100))

lights = caribu_light_sources(sun, sky)
# Build and illuminate scene
scene, labels = build_scene(g, senescence=False)
cs, raw, agg_tri = illuminate(scene=scene, light=lights, labels=labels, domain=domain, direct=False) 
agg_tri['Energy'] = agg_tri['Eabs'] * agg_tri['area']
agg_tri['Energy_i'] = agg_tri['Ei'] * agg_tri['area']
# agg_leaf = agg_tri.loc[(agg_tri.label=='Leaf'),('plant','Energy','area')].groupby('plant').agg('sum')
# agg_stem = agg_tri.loc[(agg_tri.label=='Stem'),('plant','Energy','area')].groupby('plant').agg('sum')
nrj_per_leaf = agg_tri.loc[agg_tri['label'] == 'Leaf']['Energy'].values
# nrj_per_stem = agg_tri.loc[agg_tri['label'] == 'Stem']['Energy'].values
# nrj_per_leaf_i = agg_tri.loc[agg_tri['label'] == 'Leaf']['Energy_i'].values
# nrj_per_stem_i = agg_tri.loc[agg_tri['label'] == 'Stem']['Energy_i'].values
Qi_soil, Einc_soil, Qabs_soil, Eabs_soil = getSoilEnergy(cs)

In [156]:
len(nrj_per_leaf)

20

Bilan d'énergie : somme de l'énergie incidente sur tous les organes de la plante + incident sur le sol == radiation globale 16.2

In [94]:
area_soil =((abs(domain[0][0])+domain[1][0])*(abs(domain[0][1])+domain[1][1])) / 100**2 # m2
area_soil

0.18518518518518517

In [95]:
reflected_nrj = (sum(agg_tri['Energy_i']) + Einc_soil - (sum(agg_tri['Energy']) + Eabs_soil)) / area_soil
reflected_nrj

1.137148470626593

In [96]:
absorbed_nrj = (sum(agg_tri['Energy']) + Eabs_soil) / area_soil
absorbed_nrj

7.46535107717

In [97]:
round(reflected_nrj + absorbed_nrj, 3) == par

np.False_

In [98]:
absorbed_nrj / par

np.float64(0.9600502928459362)

In [99]:
faPAR = sum(nrj_per_leaf) * density / par
faPAR

np.float64(0.7906622802763891)

In [100]:
scene_irr, values = cs.plot(raw, display=False)
PlantGL(scene_irr)

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…